# Práctica 03: Diagnóstico de Calidad de Datos (5 Dimensiones)
**Objetivo:** Auditar completitud, validez, consistencia, unicidad y atípicos.

## 1. Carga para diagnóstico

In [1]:
import glob, re, numpy as np, pandas as pd

def leer_auditoria(f):
    with open(f, 'r', encoding='latin1') as fp:
        lines = [fp.readline() for _ in range(8)]
    sep = ';' if sum(l.count(';') for l in lines) > sum(l.count(',') for l in lines) else ','
    skip = next((i for i, l in enumerate(lines) if any(k in l.upper() for k in ['LOCAL', 'DEPARTAMENTO', 'TRANSAC'])), 0)
    d = pd.read_csv(f, sep=sep, encoding='latin1', skiprows=skip)
    d.columns = [c.strip().upper() for c in d.columns]
    cmap = {}
    for c in d.columns:
        if 'LOCAL' in c or 'CENTRO' in c: cmap[c] = 'LOCAL'
        elif 'DEP' in c: cmap[c] = 'DEPARTAMENTO'
        elif 'PROV' in c: cmap[c] = 'PROVINCIA'
        elif 'DIST' in c: cmap[c] = 'DISTRITO'
        elif 'MES' in c: cmap[c] = 'MES'
        elif 'TIPO' in c: cmap[c] = 'TIPO_TRANSACCION'
        elif 'TRANSAC' in c: cmap[c] = 'TRANSACCIONES'
    d = d.rename(columns=cmap)
    d['ANIO'] = int(re.search(r'202[0-9]', f).group(0)) if re.search(r'202[0-9]', f) else 2023
    for c in ['LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES', 'TIPO_TRANSACCION', 'TRANSACCIONES']:
        if c not in d.columns: d[c] = np.nan
    return d[['LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES', 'TIPO_TRANSACCION', 'TRANSACCIONES', 'ANIO']]

archivos = sorted(glob.glob("../Dataset-project/**/*.csv", recursive=True))
df_raw = pd.concat([leer_auditoria(f) for f in archivos], ignore_index=True)
print(f"Total registros auditados: {len(df_raw):,}")

Total registros auditados: 122,527


## 2. Dimensión 1: Completitud

In [2]:
cols = ['LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES', 'TIPO_TRANSACCION', 'TRANSACCIONES']
completitud = pd.DataFrame({
    'Nulos': df_raw[cols].isnull().sum(),
    '% Faltante': (df_raw[cols].isnull().mean() * 100).round(2)
})
completitud

,Nulos,% Faltante
LOCAL,65366,53.35
DEPARTAMENTO,66291,54.10
PROVINCIA,62233,50.79
DISTRITO,68299,55.74
MES,52385,42.75
TIPO_TRANSACCION,33942,27.70
TRANSACCIONES,8802,7.18


## 3. Dimensión 2 y 3: Validez y Consistencia

In [3]:
print("Muestra meses únicos:", df_raw['MES'].dropna().unique()[:6])
print("Tipos de trámite:", df_raw['TIPO_TRANSACCION'].dropna().unique())

Muestra meses únicos: ['JULIO' 'AGOSTO' 'SEPTIEMBRE' 'OCTUBRE' 'NOVIEMBRE' 'DICIEMBRE']
Tipos de trámite: ['TRAMITE DNI' 'ENTREGA DNI' 'RRCC COPIAS CERTIF' 'CERTIFICACION DE RUIPN'
 'CERTIFICADOS DIGITALES' 'RRCC ACTA REGISTRAL' 'PAGO DE TASAS']


## 4. Dimensión 4 y 5: Unicidad y Atípicos

In [4]:
t_num = pd.to_numeric(df_raw['TRANSACCIONES'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
q1 = t_num.quantile(0.25)
q3 = t_num.quantile(0.75)
iqr = q3 - q1
outliers = (t_num > (q3 + 1.5 * iqr)).sum()
print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Outliers superiores (> Q3 + 1.5*IQR): {outliers:,}")

Q1: 14.0, Q3: 315.0, IQR: 301.0
Outliers superiores (> Q3 + 1.5*IQR): 13,708
